In [1]:
import os
from datetime import datetime, date, timedelta
from zoneinfo import ZoneInfo

from dotenv import load_dotenv

# Trading Client
from alpaca.trading.client import (
    TradingClient)     

# Trading Request Models
from alpaca.trading.requests import (
    GetCalendarRequest,
)# Instantiates a client for managing trading and personal brokerage accounts

In [2]:
# Please safely store your API keys and never commit them to the repository (use .gitignore)
# Load environment variables from environment file (e.g. .env)
load_dotenv()

TRADE_API_KEY = os.environ.get('ALPACA_API_KEY')
TRADE_API_SECRET = os.environ.get('ALPACA_SECRET_KEY')

# We use paper environment for this example
ALPACA_PAPER_TRADE = True

# Below are the variables for development this documents so please do not change these variables
TRADE_API_URL = "https://paper-api.alpaca.markets" if ALPACA_PAPER_TRADE else "https://api.alpaca.markets"
DATA_API_URL = None if ALPACA_PAPER_TRADE else "https://data.alpaca.markets"
TRADE_API_WSS = None
OPTION_STREAM_DATA_WSS = None

In [3]:
TRADE_API_KEY

'PKMDAQ4V5CCOOPF8BUUR'

In [3]:
# To enable live trading, replace `ALPACA_PAPER_TRADE = True` with `ALPACA_PAPER_TRADE = False` to disable the paper trading configuration
trade_client = TradingClient(api_key=TRADE_API_KEY, secret_key=TRADE_API_SECRET, paper=ALPACA_PAPER_TRADE)

In [4]:
# Check account related information
acc = trade_client.get_account()
acc

{   'account_blocked': False,
    'account_number': 'PA3ZPB2W362P',
    'accrued_fees': '0',
    'buying_power': '191823.1',
    'cash': '102961.55',
    'created_at': datetime.datetime(2025, 2, 7, 1, 47, 10, 815394, tzinfo=TzInfo(UTC)),
    'crypto_status': <AccountStatus.ACTIVE: 'ACTIVE'>,
    'currency': 'USD',
    'daytrade_count': 0,
    'daytrading_buying_power': '0',
    'equity': '102779.55',
    'id': UUID('94ccd725-6370-4308-b880-31e14648310a'),
    'initial_margin': '7050',
    'last_equity': '102807.55',
    'last_maintenance_margin': '7050',
    'long_market_value': '0',
    'maintenance_margin': '7050',
    'multiplier': '2',
    'non_marginable_buying_power': '95911.55',
    'options_approved_level': 3,
    'options_buying_power': '95911.55',
    'options_trading_level': 3,
    'pattern_day_trader': False,
    'pending_transfer_in': None,
    'pending_transfer_out': None,
    'portfolio_value': '102779.55',
    'regt_buying_power': '191823.1',
    'short_market_value': '

In [5]:
# Check if our account is restricted from trading.
if acc.trading_blocked: # type: ignore
    print('Account is currently restricted from trading.')

In [ ]:
acc.cash # type: ignore

'102961.55'

In [ ]:
# Check account buying power
buying_power = float(acc.buying_power) # type: ignore
buying_power

191823.1

In [8]:
# Check account configurations
acct_config = trade_client.get_account_configurations()
acct_config

{   'dtbp_check': <DTBPCheck.ENTRY: 'entry'>,
    'fractional_trading': True,
    'max_margin_multiplier': '4',
    'max_options_trading_level': None,
    'no_shorting': False,
    'pdt_check': <PDTCheck.ENTRY: 'entry'>,
    'ptp_no_exception_entry': False,
    'suspend_trade': False,
    'trade_confirm_email': <TradeConfirmationEmail.ALL: 'all'>}

In [9]:
# Get all postions 
positions = trade_client.get_all_positions()
positions

[{   'asset_class': <AssetClass.US_OPTION: 'us_option'>,
     'asset_id': UUID('97bb1b04-2966-4e4f-b70d-38b4c54cc1f1'),
     'asset_marginable': True,
     'avg_entry_price': '0.12',
     'avg_entry_swap_rate': None,
     'change_today': '0',
     'cost_basis': '-12',
     'current_price': '0.04',
     'exchange': <AssetExchange.EMPTY: ''>,
     'lastday_price': '0.04',
     'market_value': '-4',
     'qty': '-1',
     'qty_available': '-1',
     'side': <PositionSide.SHORT: 'short'>,
     'swap_rate': None,
     'symbol': 'AAL260206P00013500',
     'unrealized_intraday_pl': '0',
     'unrealized_intraday_plpc': '0',
     'unrealized_pl': '8',
     'unrealized_plpc': '0.66667',
     'usd': None},
 {   'asset_class': <AssetClass.US_OPTION: 'us_option'>,
     'asset_id': UUID('01e7fb9b-3569-4a0e-9dbc-330c908a4818'),
     'asset_marginable': True,
     'avg_entry_price': '1.65',
     'avg_entry_swap_rate': None,
     'change_today': '-0.03784',
     'cost_basis': '-165',
     'current_pri

In [10]:
# Check the cost basis and unrealised profit/loss of the open position
# Show cost basis
if isinstance(positions, list):
    if len(positions) == 0:
        print("No open positions found.")
    else:
        # Show unrealized profit/loss
        for pos in positions:
            print(pos.unrealized_pl)

8
-13


In [39]:
# cal_request = GetCalendarRequest(start = date.today() + timedelta(days=3), end = date.today() + timedelta(days=3))
cal_request = GetCalendarRequest(start = date.today(), end = date.today())
cal = trade_client.get_calendar(cal_request)
if cal:
    print("True")
else:
    print("False")
cal


True


[{   'close': datetime.datetime(2026, 2, 6, 16, 0),
     'date': datetime.date(2026, 2, 6),
     'open': datetime.datetime(2026, 2, 6, 9, 30)}]

In [38]:
clock_request = ()
clock = trade_client.get_clock()
clock

{   'is_open': False,
    'next_close': datetime.datetime(2026, 2, 6, 16, 0, tzinfo=TzInfo(-05:00)),
    'next_open': datetime.datetime(2026, 2, 6, 9, 30, tzinfo=TzInfo(-05:00)),
    'timestamp': datetime.datetime(2026, 2, 5, 18, 7, 57, 984927, tzinfo=TzInfo(-05:00))}

In [40]:
clock.next_open.hour + 1

10

In [26]:
clock.next_close.hour

16

In [ ]:
def is_trading_window() -> bool:
    current_time_ny = datetime.now(tz=ZoneInfo("America/New_York"))
    is_weekday = current_time_ny.weekday() < 5  # Monday=0, Sunday=6
    is_after_930am = current_time_ny.hour > clock.next_open.hour + 1 or (current_time_ny.hour == clock.next_open.hour and current_time_ny.minute >= clock.next_open.minute)
    is_before_4pm = current_time_ny.hour < clock.next_close.hour
    # Use US holiday API or static list for demonstration
    us_holidays = [
        "2025-01-01", "2025-01-20", "2025-02-17", "2025-04-18", "2025-05-26",
        "2025-19-06", "2025-07-04", "2025-09-01", "2025-11-27", "2025-12-25"
    ]
    date_str = current_time_ny.strftime("%Y-%m-%d")
    is_us_holiday = date_str in us_holidays

    is_trading_window = is_weekday and is_after_930am and is_before_4pm  and not is_us_holiday

    return is_trading_window

print(f"Is trading window open? {is_trading_window()}")